# Linear drag (Taylor 2.1-2.2)

*Class notebook, PHY 317.* Run each cell in order with **Shift-Enter**.

Drag on a sphere of diameter $D$ moving at speed $v$ through a fluid is
$$f(v) = b v + c v^2, \qquad b = \beta D, \quad c = \gamma D^2,$$
with $\beta = 1.6\times10^{-4}\ \mathrm{N\,s/m^2}$ and $\gamma = 0.25\ \mathrm{N\,s^2/m^4}$ for air (Taylor Eqs. 2.4-2.6). The first question about any projectile is which term matters.

## 1. Predict first (before you run anything)

Double-click the cell below, type, Shift-Enter.

- For a thrown baseball, which term dominates: $bv$ (linear) or $cv^2$ (quadratic)? For a falling raindrop? For a speck of dust?
- With **linear** drag only, a projectile launched horizontally: does it travel a finite or an infinite horizontal distance before it has fallen forever?

*(your prediction here)*

## 2. Setup

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

g = 9.8                       # m/s^2
beta, gamma = 1.6e-4, 0.25    # air at STP, SI units

## 3. Which drag term matters?

The ratio of the two terms is $f_{\rm quad}/f_{\rm lin} = (\gamma/\beta)\,D v \approx (1.6\times10^3\ \mathrm{s/m^2})\,D v$.

In [ ]:
objects = [("baseball", 0.07, 5.0), ("raindrop", 1e-3, 0.6), ("oil drop", 1.5e-6, 5e-5)]

for name, D, v in objects:
    ratio = gamma / beta * D * v
    print(f"{name:9s}  D = {D:8.1e} m  v = {v:7.1e} m/s   f_quad / f_lin = {ratio:.1e}")

## 4. Linear drag, solved

With $y$ up, $\tau = m/b$ and $v_{\rm ter} = mg/b$:
$$v_x = v_{x0}e^{-t/\tau}, \qquad x = v_{x0}\tau\,(1 - e^{-t/\tau}),$$
$$v_y = -v_{\rm ter} + (v_{y0} + v_{\rm ter})e^{-t/\tau}, \qquad y = -v_{\rm ter}\,t + (v_{y0} + v_{\rm ter})\,\tau\,(1 - e^{-t/\tau}).$$
Things to look for as you move $\tau$: $v_x$ drops to 37% after one $\tau$; $x$ never passes $v_{x0}\tau$ (dotted line); $v_y$ is within 5% of $-v_{\rm ter}$ after $3\tau$.

In [ ]:
def linear_drag(t, tau, vx0, vy0):
    """Position and velocity under linear drag, y up. Returns x, y, vx, vy."""
    vter = g * tau
    decay = np.exp(-t / tau)
    vx = vx0 * decay
    x = vx0 * tau * (1 - decay)
    vy = -vter + (vy0 + vter) * decay
    y = -vter * t + (vy0 + vter) * tau * (1 - decay)
    return x, y, vx, vy


@interact(tau=(0.1, 10.0, 0.1), vx0=(0, 30, 1), vy0=(-20, 30, 1))
def show(tau=2.0, vx0=10, vy0=10):
    t = np.linspace(0, 8, 500)
    x, y, vx, vy = linear_drag(t, tau, vx0, vy0)
    vter = g * tau

    fig, ax = plt.subplots(2, 2, figsize=(9, 5.5))
    ax[0, 0].plot(t, vx)
    ax[0, 0].set_title("v_x (m/s)")
    ax[0, 1].plot(t, x)
    ax[0, 1].axhline(vx0 * tau, linestyle=":", color="gray")
    ax[0, 1].set_title("x (m)   dotted: v_x0 tau")
    ax[1, 0].plot(t, vy)
    ax[1, 0].axhline(-vter, linestyle=":", color="gray")
    ax[1, 0].set_title("v_y (m/s)   dotted: -v_ter")
    ax[1, 1].plot(t, y)
    ax[1, 1].set_title("y (m)")
    for a in ax.flat:
        a.grid(alpha=0.3)
    for a in ax[1]:
        a.set_xlabel("t (seconds)")
    plt.tight_layout()
    plt.show()

## 5. What is $\tau$ for a real drop?

Stokes's law gives $b = 3\pi\eta D$, so $\tau = m/b$ with $m = \rho_{\rm obj}\,\pi D^3/6$. Try Taylor's examples: a Millikan oil drop ($D = 1.5\ \mu$m, $\rho = 840$ kg/m$^3$) and a 0.2 mm water drop.

In [ ]:
viscosity = {"air": 1.7e-5, "water": 1.0e-3}    # Pa s

@interact(D_mm=(0.001, 2.0, 0.001), rho_obj=(500, 8000, 100), medium=["air", "water"])
def stokes(D_mm=0.2, rho_obj=1000, medium="air"):
    D = D_mm * 1e-3
    m = rho_obj * np.pi * D**3 / 6
    b = 3 * np.pi * viscosity[medium] * D
    tau = m / b
    print(f"m = {m:.2e} kg    b = {b:.2e} kg/s    tau = {tau:.2e} s    v_ter = {m * g / b:.2e} m/s")

## 6. What you found

Double-click, type, Shift-Enter. How did it compare with your prediction?

*(what you found, compared with your prediction)*